In [205]:
import re
import requests
import pandas as pd


from bs4 import BeautifulSoup

In [206]:
url = "https://finance.yahoo.com/quote/GOOG/history"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
response = requests.get(url, headers=headers)
print(response.status_code)

200


In [207]:
soup = BeautifulSoup(response.content, 'html.parser')

In [208]:
page_elements = soup.find_all('td', class_='yf-1jecxey')

In [209]:
page_elements = [element.text for element in page_elements]

In [210]:
def remove_non_stock_data(input_list: list) -> list:
    cleaned_list = []
    for item in input_list:
        date_match = bool(re.match(r"\w{3} \d{1,2}, \d{4}", item))
        stock_match = bool(re.match(r"^\d{1,3}(,\d{3})*(\.\d{2})?$", item))
        vol_match = bool(re.match(r"^\d{1,3}(,\d{3})+$", item))

        if not (date_match or  stock_match or  vol_match):
            print(f"non stock data found : {item} on {i} index")
            cleaned_list = cleaned_list[:-1]
            continue

        cleaned_list.append(item)

    return cleaned_list


def parse_data(input_list):
    parsed_data = []
    for i in range(0, len(input_list), 7):
        if i+6 < len(input_list):
            temp_dict = {
                "Date": input_list[i],
                "Open": input_list[i+1],
                "High": input_list[i+2],
                "Low": input_list[i+3],
                "Close": input_list[i+4],
                "Volume": input_list[i+6]
            }
            parsed_data.append(temp_dict)
        else:
            print(f"Skipping incomplete data at index {i}")
    return parsed_data

In [211]:
page_elements = remove_non_stock_data(page_elements)

non stock data found : 0.2 Dividend  on 249 index
non stock data found : 0.2 Dividend  on 249 index
non stock data found : 0.2 Dividend  on 249 index
non stock data found : 0.2 Dividend  on 249 index


In [212]:
parsed_data_stock = parse_data(page_elements)

In [213]:
df = pd.DataFrame(parsed_data_stock)

In [214]:
df.head()

,Date,Open,High,Low,Close,Volume
0,"May 2, 2025",164.96,166.70,163.66,165.81,"16,832,500"
1,"May 1, 2025",162.52,163.94,160.93,162.79,"21,904,300"
2,"Apr 30, 2025",159.86,161.37,157.15,160.89,"20,639,500"
3,"Apr 29, 2025",162.04,162.68,159.39,162.06,"15,955,200"
4,"Apr 28, 2025",164.26,164.95,160.38,162.42,"20,871,200"


In [215]:
df.tail()

,Date,Open,High,Low,Close,Volume
245,"May 9, 2024",171.15,172.44,169.93,171.58,"11,937,700"
246,"May 8, 2024",170.75,171.91,170.52,171.16,"14,569,900"
247,"May 7, 2024",170.12,173.47,170.00,172.98,"21,102,400"
248,"May 6, 2024",169.22,169.90,167.89,169.83,"15,147,900"
249,"May 3, 2024",169.54,169.85,164.98,168.99,"22,767,100"


In [216]:
df.tail()

,Date,Open,High,Low,Close,Volume
245,"May 9, 2024",171.15,172.44,169.93,171.58,"11,937,700"
246,"May 8, 2024",170.75,171.91,170.52,171.16,"14,569,900"
247,"May 7, 2024",170.12,173.47,170.00,172.98,"21,102,400"
248,"May 6, 2024",169.22,169.90,167.89,169.83,"15,147,900"
249,"May 3, 2024",169.54,169.85,164.98,168.99,"22,767,100"


In [217]:
for i in range(df.shape[0]):
    string = df['Date'][i]
    match = re.match(r"(\w{3}) (\d{1,2}), (\d{4})", string)
    if not match:
        print(i)

In [218]:
def date_format(input:str):
    input = input.lower()
    month_to_number = {
    'jan': '01', 'feb': '02', 'mar': '03',
    'apr': '04', 'may': '05', 'jun': '06',
    'jul': '07', 'aug': '08', 'sep': '09',
    'oct': '10', 'nov': '11', 'dec': '12'
    }
    match = re.match(r"(\w{3}) (\d{1,2}), (\d{4})", input)
    if match:
        month_str, day, year = match.groups()
        if len(day) < 2:
            day = '0' + str(day)
        month_num = month_to_number.get(month_str, '00')
    
    return f"{year}-{month_num}-{day}"

In [219]:
df['Date'] = df['Date'].apply(date_format)

In [220]:
df['Date']

0      2025-05-02
1      2025-05-01
2      2025-04-30
3      2025-04-29
4      2025-04-28
          ...    
245    2024-05-09
246    2024-05-08
247    2024-05-07
248    2024-05-06
249    2024-05-03
Name: Date, Length: 250, dtype: object

In [221]:
old_stock = pd.read_csv("../data/aapl_stock_price.csv", index_col=[0], parse_dates=[0])

In [222]:
old_stock.tail()

,Close,High,Low,Open,Volume
Date,,,,,
2025-04-28,210.14,211.50,207.46,210.00,"38,743,100"
2025-04-29,211.21,212.24,208.37,208.69,"36,827,600"
2025-04-30,212.50,213.58,206.67,209.30,"52,286,500"
2025-05-01,213.32,214.56,208.90,209.08,"57,365,700"
2025-05-02,205.35,206.99,202.16,206.09,"100,912,500"


In [223]:
print(old_stock[old_stock.index.year == 2025].shape)
old_stock[old_stock.index.year == 2025]

(83, 5)


,Close,High,Low,Open,Volume
Date,,,,,
2025-01-02,243.850006,249.100006,241.820007,248.929993,55740700
2025-01-03,243.360001,244.179993,241.889999,243.360001,40244100
2025-01-06,245.000000,247.330002,243.199997,244.309998,45045600
2025-01-07,242.210007,245.550003,241.350006,242.979996,40856000
2025-01-08,242.699997,243.710007,240.050003,241.919998,37628900
...,...,...,...,...,...
2025-04-28,210.140000,211.500000,207.460000,210.000000,"38,743,100"
2025-04-29,211.210000,212.240000,208.370000,208.690000,"36,827,600"
2025-04-30,212.500000,213.580000,206.670000,209.300000,"52,286,500"


In [224]:
df['Date'] = pd.to_datetime(df['Date'])

In [225]:
df = df[df['Date'] > "2025-01-21"]
df = df.set_index("Date")

In [226]:
df.head()

,Open,High,Low,Close,Volume
Date,,,,,
2025-05-02,164.96,166.70,163.66,165.81,"16,832,500"
2025-05-01,162.52,163.94,160.93,162.79,"21,904,300"
2025-04-30,159.86,161.37,157.15,160.89,"20,639,500"
2025-04-29,162.04,162.68,159.39,162.06,"15,955,200"
2025-04-28,164.26,164.95,160.38,162.42,"20,871,200"


In [227]:
df = df.sort_index()
df.head()

,Open,High,Low,Close,Volume
Date,,,,,
2025-01-22,200.55,202.12,199.20,200.03,"15,477,400"
2025-01-23,199.98,201.94,196.82,199.58,"15,170,800"
2025-01-24,199.85,202.57,199.78,201.90,"12,732,400"
2025-01-27,194.19,198.67,192.70,193.77,"24,970,200"
2025-01-28,194.65,197.23,192.61,197.07,"15,939,200"


In [228]:
new_data = pd.concat([old_stock, df], axis=0)

In [229]:
new_data.to_csv("../data/goog_stock_price.csv")